In [0]:
spark.conf.set(
  "fs.azure.account.key.cmsstorageaccdiv.dfs.core.windows.net",
  "<storage-account-key>"
)

In [0]:
dbutils.fs.ls("abfss://bronze@cmsstorageaccdiv.dfs.core.windows.net/")

[FileInfo(path='abfss://bronze@cmsstorageaccdiv.dfs.core.windows.net/cms/', name='cms/', size=0, modificationTime=1781343033000)]

In [0]:
display(dbutils.fs.ls("abfss://silver@cmsstorageaccdiv.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://silver@cmsstorageaccdiv.dfs.core.windows.net/cms/,cms/,0,1781367161000


In [0]:
dbutils.fs.ls("abfss://silver@cmsstorageaccdiv.dfs.core.windows.net/")

[FileInfo(path='abfss://silver@cmsstorageaccdiv.dfs.core.windows.net/cms/', name='cms/', size=0, modificationTime=1781367161000)]

In [0]:
df_silver = spark.read.parquet(
    "abfss://silver@cmsstorageaccdiv.dfs.core.windows.net/cms/"
)

In [0]:
display(df_silver.limit(5))

aggregation_level,average_number_of_providers_per_cbsa,average_number_of_users_per_provider,cbsa,cbsatitle,number_of_dual_eligible_users,number_of_fee_for_service_beneficiaries,number_of_providers,number_of_users,percentage_of_dual_eligible_users_out_of_dual_eligible_ffs_beneficiaries,percentage_of_dual_eligible_users_out_of_total_users,percentage_of_users_out_of_ffs_beneficiaries,reference_period,total_payment,type_of_service,load_date
NATION + TERRITORIES,116.56,424.34,--ALL--,--ALL--,1085184,37359009,9078,3852199,17.24%,28.17%,10.31%,2015-01-01 to 2015-12-31,3.430203621E9,Ambulance (Emergency & Non-Emergency),2026-06-13
CBSA,3,231,10100,"Aberdeen, SD",174,7526,3,693,18.73%,25.11%,9.21%,2015-01-01 to 2015-12-31,588150.97,Ambulance (Emergency & Non-Emergency),2026-06-13
CBSA,18,110.44,10140,"Aberdeen, WA",675,17349,18,1988,19.55%,33.95%,11.46%,2015-01-01 to 2015-12-31,1786700.58,Ambulance (Emergency & Non-Emergency),2026-06-13
CBSA,11,271.09,10180,"Abilene, TX",907,25198,11,2982,19.76%,30.42%,11.83%,2015-01-01 to 2015-12-31,2003346.18,Ambulance (Emergency & Non-Emergency),2026-06-13
CBSA,5,104.8,10220,"Ada, OK",183,7570,5,524,12.12%,34.92%,6.92%,2015-01-01 to 2015-12-31,359960.14,Ambulance (Emergency & Non-Emergency),2026-06-13


Payment by Service

In [0]:
from pyspark.sql.functions import sum

gold_payment = (
    df_silver
    .groupBy("type_of_service")
    .agg(sum("total_payment").alias("total_payment"))
)

display(gold_payment)

type_of_service,total_payment
Ambulance (Emergency),2.3903495136E9
Ambulance (Emergency & Non-Emergency),6.860407241689992E9


In [0]:
gold_payment.write.mode("overwrite").parquet(
    "abfss://gold@cmsstorageaccdiv.dfs.core.windows.net/payment_by_service/"
)

Users by Service

In [0]:
gold_users = (
    df_silver
    .groupBy("type_of_service")
    .agg(sum("number_of_users").alias("total_users"))
)

display(gold_users)

type_of_service,total_users
Ambulance (Emergency),3659973
Ambulance (Emergency & Non-Emergency),7704398


In [0]:
gold_users.write.mode("overwrite").parquet(
    "abfss://gold@cmsstorageaccdiv.dfs.core.windows.net/users_by_service/"
)

Provider Analysis

In [0]:
gold_providers = (
    df_silver
    .groupBy("type_of_service")
    .agg(sum("number_of_providers").alias("total_providers"))
)

display(gold_providers)

type_of_service,total_providers
Ambulance (Emergency),9530
Ambulance (Emergency & Non-Emergency),24939


In [0]:
gold_payment.count()
gold_users.count()

2